# Student Performance Model Training

This notebook prepares the data for modeling, trains a classification model, evaluates its performance, and saves the trained artifact for later use.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')

DATA_PATH = os.path.join('src', 'notebook', 'data', 'student_performance_data.csv')
MODEL_PATH = os.path.join('src', 'artifacts', 'student_performance_model.joblib')

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
df.head()

## Data Preparation

This section prepares the features and target for training. Categorical columns are encoded, numerical columns are scaled, and missing values are handled automatically.

In [ ]:
target_column = 'grade'
feature_columns = [
    'gender', 'study_hours_per_day', 'attendance_percentage', 'assignment_score',
    'midterm_score', 'final_exam_score', 'participation_score', 'internet_access',
    'extra_classes', 'parent_education', 'sleep_hours'
]

X = df[feature_columns]
y = df[target_column]

categorical_features = ['gender', 'internet_access', 'extra_classes', 'parent_education']
numerical_features = [col for col in feature_columns if col not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## Model Training

A Random Forest classifier is trained on the prepared data. It is a good first choice for structured tabular classification tasks and is easy to interpret.

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])

model.fit(X_train, y_train)
print('Model training completed successfully.')

## Model Evaluation

The trained model is evaluated on the holdout test set using accuracy and a detailed classification report.

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print('Accuracy:', round(accuracy, 4))
print('\nClassification Report:\n')
print(classification_report(y_test, predictions))

cm = confusion_matrix(y_test, predictions, labels=np.unique(y_test))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Save Trained Model

The trained pipeline is saved to disk so it can be reused in later prediction workflows.

In [ ]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(model, MODEL_PATH)
print('Model saved at:', MODEL_PATH)